In [1]:
import torch

UP = 0
DOWN = 1
LEFT = 2
RIGHT = 3




class Gridworld:
    def __init__(self, device, no_runs, gridsize, start_state=(5, 3), goal_state=(0, 8)) -> None:
        self.device = device
        self.no_runs = no_runs
        self.runs =  torch.arange(self.no_runs)

        # --- Static ENV for simplicity ---
        self.gridsize = gridsize
        self.max_row_idx = self.gridsize[0] - 1
        self.max_col_idx = self.gridsize[1] - 1


        self.start_state = torch.tensor(start_state, device=self.device)
        self.goal_state =  torch.tensor(goal_state, device=self.device)
        self.actions = torch.tensor((0,1,2,3), device=self.device)

        self.wall = torch.zeros(self.gridsize,device=device, dtype=torch.bool)
        self.state = torch.tile(self.start_state, (self.no_runs,1))

    def step(self, action):
        prev_state = self.state.clone()

        # update state for up, down, left, right
        self.state[self.runs,0] -= (action[self.runs] == 0).int()
        self.state[self.runs,0] += (action[self.runs] == 1  ).int()
        self.state[self.runs,1] -= (action[self.runs] == 2 ).int()
        self.state[self.runs,1] += (action[self.runs] == 3).int()

        # do not let agent exit grid
        self.state[self.state[self.runs, 0] > self.max_row_idx, 0] = self.max_row_idx
        self.state[self.state[self.runs, 1] > self.max_col_idx, 1] = self.max_col_idx

        self.state[self.state[self.runs, 0] < 0, 0] = 0
        self.state[self.state[self.runs, 1] < 0, 1] = 0

        # wall
        mask = self.wall[self.state[self.runs, 0], self.state[self.runs, 1]]
        self.state[mask] = prev_state[mask]

        # Do not let agent leave terminal state
        terminal_mask = (prev_state[self.runs] == self.goal_state).all(dim=-1) 
        self.state[terminal_mask] = self.goal_state

        # Compute rewards
        mask = (self.state[self.runs] == self.goal_state).all(dim=-1) &  ~terminal_mask
        reward = mask.int()

        processed_state = self.get_processed_state()
        return processed_state, reward, terminal_mask
    
    def get_processed_state(self):
        # State is a number 0 - (6 * 9 - 1)
        return self.state[:, 0] * 9 + self.state[:, 1]


    def set_wall(self, index):
        self.wall[index] = True

    def reset_wall(self):
        self.wall = torch.zeros(self.gridsize, device=self.device)


    def reset(self):
        self.reset_wall()
        self.state = torch.tile(self.start_state, (self.no_runs, 1))  
        return self.start_state 
    

            
            





gridsize = 6,9
no_runs = 10

env = Gridworld("cpu", no_runs, gridsize)

env.step(torch.full((no_runs,), UP))

env.step(torch.full((no_runs,), RIGHT))




# torch.multinomial(weights, 4, replacement=True)


(tensor([40, 40, 40, 40, 40, 40, 40, 40, 40, 40]),
 tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=torch.int32),
 tensor([False, False, False, False, False, False, False, False, False, False]))

In [2]:
class DynaQ_plus:
    def __init__(self,device, no_runs, no_states, no_actions=4 , kappa= 0.1, alpha= 0.1, gamma = 0.95) -> None:
        self.no_runs = no_runs
        self.runs = torch.arange(no_runs)

        self.Q = torch.zeros((no_runs,no_states,no_actions), device=device)
        
        self.kappa = kappa
        self.alpha = alpha
        self.gamma = gamma
        
        self.visited = torch.empty((no_runs, 0,2), dtype=torch.long, device=device) # state, action
        self.model = torch.full((no_runs,no_states,no_actions,2), -1, device=device) # (next state, reward)

        self.device = device
    
    def train(self, episodes,planning_steps,env: Gridworld):
        for i in range(episodes):
            # encode the state so that it is (row_idx * no_cols) + col_idx
            state = env.get_processed_state()

            print(state.shape)

            action = self.Q[self.runs,state].argmax(dim=-1)
            next_state, r, done = env.step(action)
            
            self.Q[self.runs,state, action] += self.alpha  * (r + self.gamma * self.Q[self.runs, next_state].max(dim=-1).values)

            
agent = DynaQ_plus("cpu", no_runs,6 * 9,4)


agent.train(1 ,5, env)

torch.Size([10])
